In [2]:
import pandas as pd
import numpy as np
import os
from datetime import datetime
import glob

In [ ]:
# data_folder_path = "/home/hsph/Downloads/DataStore/SS/ASIANPAINT/2024/"
# csv_path_list = glob.glob(data_folder_path + "*.csv")

In [4]:
def process_csv(csv_path): 
# Load the CSV file
    df = pd.read_csv(csv_path)

    # Convert 'Date Time' column to datetime if it's not already
    df['Date Time'] = pd.to_datetime(df['Date Time'])

    # Extract the date of the file (assumes all rows in the file are from the same day)
    file_date = df['Date Time'].dt.date.iloc[0]

    # Get the opening spot price (first row value of 'Spot')
    opening_spot = df.loc[0, 'Spot']

    # Calculate moneyness
    df['moneyness'] = df['Strike'] / opening_spot

    # Adjust moneyness for 'PE' type options
    df.loc[df['Type'] == 'PE', 'moneyness'] = 1 / df['moneyness']

    # Moneyness in percentage terms
    df['moneyness'] = (df['moneyness'] - 1) * 100

    # Spread in percentage
    df['spread_pct'] = df['bid_ask_spread'] / df['mid_price'] * 100

    # Define the bucketing function
    def bucket_moneyness(value):
        if value >= 0.5:
            return min(int(np.floor(value - 0.5) + 1), 11)  # Bucketing for positive values
        elif value <= -0.5:
            return max(int(np.ceil(value + 0.5) - 1), -11)  # Bucketing for negative values
        else:
            return 0  # Values between -0.5 and 0.5 are set to 0

    # Apply bucketing
    df['moneyness_bucket'] = df['moneyness'].apply(bucket_moneyness)

    # Function to calculate liquidity
    def calculate_liquidity(group):
        return (1 - (group['BidPrice'].diff() == 0).sum() / len(group)) * 100

    # Function to calculate spread metrics (including avg_spread_value)
    def calculate_spread_metrics(group):
        max_idx = group['spread_pct'].idxmax()
        min_idx = group['spread_pct'].idxmin()
        
        return pd.Series({
            'max_spread_value': group.loc[max_idx, 'spread_pct'],
            'max_spread_time': group.loc[max_idx, 'Date Time'].time(),
            'min_spread_value': group.loc[min_idx, 'spread_pct'],
            'min_spread_time': group.loc[min_idx, 'Date Time'].time(),
            'avg_spread_value': group['spread_pct'].mean()  # Average spread value
        })

    # Function to calculate price problem percentage
    def calculate_price_problem_pct(group):
        return (group['price_problem'] == True).sum() / len(group)

    # Group by 'moneyness_bucket' and 'Type' and compute metrics
    # Compute group metrics with include_groups=False to avoid the DeprecationWarning
    liquidity = df.groupby(['moneyness_bucket', 'Type']).apply(calculate_liquidity, include_groups=False).reset_index(name='liquidity')
    spread_metrics = df.groupby(['moneyness_bucket', 'Type']).apply(calculate_spread_metrics, include_groups=False).reset_index()
    price_problem_pct = df.groupby(['moneyness_bucket', 'Type']).apply(calculate_price_problem_pct, include_groups=False).reset_index(name='price_problem_pct')


    # Merge all computed values
    result = liquidity.merge(spread_metrics, on=['moneyness_bucket', 'Type']).merge(price_problem_pct, on=['moneyness_bucket', 'Type'])

    # Convert to dictionary with dynamic variable names
    result_dict = {'Date': file_date}  # Add the Date column

    for _, row in result.iterrows():
        key_prefix = f"{row['Type']}_{row['moneyness_bucket']}"
        
        result_dict[f"liq_{key_prefix}"] = row['liquidity']
        result_dict[f"spread_{key_prefix}_max_value"] = row["max_spread_value"]
        result_dict[f"spread_{key_prefix}_max_time"] = row["max_spread_time"]
        result_dict[f"spread_{key_prefix}_min_value"] = row["min_spread_value"]
        result_dict[f"spread_{key_prefix}_min_time"] = row["min_spread_time"]
        result_dict[f"spread_{key_prefix}_avg_value"] = row["avg_spread_value"]
        result_dict[f"price_problem_pct_{key_prefix}"] = row["price_problem_pct"]

    return result_dict



In [5]:
# all_results = []

# for file_path in csv_path_list:
#     daily_result = process_csv(file_path)
#     all_results.append(daily_result)

# # Convert list of dictionaries into a DataFrame
# final_df = pd.DataFrame(all_results)

# # Convert 'Date' column to datetime format (ensures correct sorting)
# final_df['Date'] = pd.to_datetime(final_df['Date'])

# # Sort DataFrame by 'Date' in ascending order (earliest to latest)
# final_df = final_df.sort_values(by='Date', ascending=True).reset_index(drop=True)

# # Save the result as a CSV (optional)
# final_df.to_csv("/home/hsph/Downloads/DataStore/SS/ASIANPAINT/2024/data_report.csv", index=False)

# # Display the final DataFrame
# final_df.head()

## Processing multiple Stocks in one go

In [ ]:
# folder_path_list = glob.glob("/home/hsph/Downloads/OneDrive_3_4-6-2025/*/2024/2024_new/")
folder_path_list = glob.glob("/home/hsph/Downloads/OneDrive_4_4-6-2025/*/*/*/")

for folder_path in folder_path_list:
    csv_path_list = glob.glob(folder_path + "*.csv")
    stock_name = folder_path.split('/')[-4]
    all_results = []

    for file_path in csv_path_list:
        daily_result = process_csv(file_path)
        all_results.append(daily_result)

    # Convert list of dictionaries into a DataFrame
    final_df = pd.DataFrame(all_results)

    # Convert 'Date' column to datetime format (ensures correct sorting)
    final_df['Date'] = pd.to_datetime(final_df['Date'])

    # Sort DataFrame by 'Date' in ascending order (earliest to latest)
    final_df = final_df.sort_values(by='Date', ascending=True).reset_index(drop=True)

    # Save the result as a CSV (optional)
    final_df.to_csv(f"/home/hsph/Downloads/DataStore/SS/data_reports/2024/{stock_name}.csv", index=False)

    print("Report Generated : ", stock_name)

# Tasks
### Liquidity using BID-ASK movement (median threshold)
### Using Volume based approach - Last Traded Volume and Last traded Price
### Price Filter : Step by Step : See if the LTP between Bid and Ask : If volume is not zero (ie. LTP is also non zero)
### DDG Approach : See the stream of BID/ASK prices : If they are sustained for some time, then they are reasonable, otherwise they are not